In [1]:
# CELL 1

import os
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report, f1_score, balanced_accuracy_score

warnings.filterwarnings("ignore")

try:
    import cv2
except Exception:
    cv2 = None

from tqdm import tqdm
import torchvision

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cv2:", "OK" if cv2 is not None else "NOT AVAILABLE")


torch: 2.5.1+cu121
torchvision: 0.20.1+cu121
cv2: OK


In [2]:
# CELL 2

@dataclass
class CFG:
    BASE_DIR: Path = Path(r"F:\Bracu\THESIS\Final Defence\Dataset\code")
    CSV_NAME: str = "dmsa dataset.csv"

    COL_PATIENT: str = "PATIENT NAME"
    COL_LINK: str = "LINK"
    COL_VIEW: str = "VIEW"

    COL_KIDNEY_COUNT: str = "Kidney COUNT"
    COL_ANATOMY: str = "ANATOMY"
    COL_POSITION: str = "POSITION"
    COL_SCAR: str = "SCAR"
    COL_UP_L: str = "CORTICAL UPTAKE LEFT"
    COL_UP_R: str = "CORTICAL UPTAKE RIGHT"

    IMG_SIZE: int = 256

    SEED: int = 42
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Phase-2 checkpoint (pretrained backbone + anatomy/position heads)
    PHASE2_CKPT: str = "phase2_count_anat_pos_best.pt"

    # ========= Preprocessing =========
    APPLY_LAYOUT_CROP: bool = True
    CROP_TOP_RATIO: float = 0.14
    CROP_RIGHT_RATIO: float = 0.14
    CROP_LEFT_RATIO: float = 0.00
    CROP_BOTTOM_RATIO: float = 0.00

    BODY_Q: float = 70.0
    BODY_MIN_PIXELS: int = 250

    INVERT_INTENSITY: bool = True
    APPLY_CLAHE: bool = True
    CLAHE_CLIP: float = 2.0
    CLAHE_TILE: int = 8
    APPLY_DENOISE: bool = True

    AUTO_ROI_CROP: bool = True
    HOT_Q: float = 88.0
    HOT_MIN_AREA: int = 80
    HOT_MAX_KEEP: int = 2
    BLADDER_CY_FRAC: float = 0.72
    THIN_TALL_W: int = 6
    THIN_TALL_H_FRAC: float = 0.25
    ROI_PAD_RATIO: float = 0.18

    # ========= Training =========
    BATCH_SIZE: int = 16
    LR: float = 3e-5
    WEIGHT_DECAY: float = 1e-4
    EPOCHS: int = 12
    PATIENCE: int = 4

    # multi-task loss weights (count remains dominant)
    W_ANATOMY: float = 0.35
    W_POSITION: float = 0.25
    W_SCAR: float = 0.35
    W_UP_L: float = 0.25
    W_UP_R: float = 0.25

cfg = CFG()
print(cfg)


CFG(BASE_DIR=WindowsPath('F:/Bracu/THESIS/Final Defence/Dataset/code'), CSV_NAME='dmsa dataset.csv', COL_PATIENT='PATIENT NAME', COL_LINK='LINK', COL_VIEW='VIEW', COL_KIDNEY_COUNT='Kidney COUNT', COL_ANATOMY='ANATOMY', COL_POSITION='POSITION', COL_SCAR='SCAR', COL_UP_L='CORTICAL UPTAKE LEFT', COL_UP_R='CORTICAL UPTAKE RIGHT', IMG_SIZE=256, SEED=42, DEVICE='cuda', PHASE2_CKPT='phase2_count_anat_pos_best.pt', APPLY_LAYOUT_CROP=True, CROP_TOP_RATIO=0.14, CROP_RIGHT_RATIO=0.14, CROP_LEFT_RATIO=0.0, CROP_BOTTOM_RATIO=0.0, BODY_Q=70.0, BODY_MIN_PIXELS=250, INVERT_INTENSITY=True, APPLY_CLAHE=True, CLAHE_CLIP=2.0, CLAHE_TILE=8, APPLY_DENOISE=True, AUTO_ROI_CROP=True, HOT_Q=88.0, HOT_MIN_AREA=80, HOT_MAX_KEEP=2, BLADDER_CY_FRAC=0.72, THIN_TALL_W=6, THIN_TALL_H_FRAC=0.25, ROI_PAD_RATIO=0.18, BATCH_SIZE=16, LR=3e-05, WEIGHT_DECAY=0.0001, EPOCHS=12, PATIENCE=4, W_ANATOMY=0.35, W_POSITION=0.25, W_SCAR=0.35, W_UP_L=0.25, W_UP_R=0.25)


In [3]:
# CELL 3

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.SEED)

csv_path = cfg.BASE_DIR / cfg.CSV_NAME
df = pd.read_csv(csv_path)

if "Unnamed: 23" in df.columns:
    df = df.drop(columns=["Unnamed: 23"])

# present-only for all phases now
df = df[df[cfg.COL_KIDNEY_COUNT].isin([1, 2])].reset_index(drop=True)

needed = [
    cfg.COL_PATIENT, cfg.COL_LINK, cfg.COL_VIEW,
    cfg.COL_KIDNEY_COUNT, cfg.COL_ANATOMY, cfg.COL_POSITION,
    cfg.COL_SCAR, cfg.COL_UP_L, cfg.COL_UP_R
]
for c in needed:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

print("DF shape:", df.shape)
print("Kidney COUNT:\n", df[cfg.COL_KIDNEY_COUNT].value_counts().sort_index())
print("SCAR unique:", sorted(df[cfg.COL_SCAR].astype(str).str.strip().str.upper().unique().tolist()))
print("UP_L unique:", sorted(df[cfg.COL_UP_L].astype(str).str.strip().str.upper().unique().tolist()))
print("UP_R unique:", sorted(df[cfg.COL_UP_R].astype(str).str.strip().str.upper().unique().tolist()))

df[[cfg.COL_PATIENT, cfg.COL_LINK, cfg.COL_VIEW, cfg.COL_KIDNEY_COUNT, cfg.COL_ANATOMY, cfg.COL_POSITION, cfg.COL_SCAR, cfg.COL_UP_L, cfg.COL_UP_R]].head()


DF shape: (2294, 23)
Kidney COUNT:
 Kidney COUNT
1     443
2    1851
Name: count, dtype: int64
SCAR unique: ['NO', 'YES']
UP_L unique: ['NORMAL', 'NOT APPLICABLE', 'REDUCED']
UP_R unique: ['NORMAL', 'NOT APPLICABLE', 'REDUCED']


,PATIENT NAME,LINK,VIEW,Kidney COUNT,ANATOMY,POSITION,SCAR,CORTICAL UPTAKE LEFT,CORTICAL UPTAKE RIGHT
0,A RAHMAN,Data\A_RAHMAN_6_MONTHS.jpeg,P,2,NORMAL,NORMAL RENAL FOSSA,YES,NORMAL,REDUCED
1,A RAHMAN,Data\A_RAHMAN_6_MONTHS_1.jpeg,LPO,2,NORMAL,NORMAL RENAL FOSSA,YES,NORMAL,REDUCED
2,A RAHMAN,Data\A_RAHMAN_6_MONTHS_2.jpeg,RPO,2,NORMAL,NORMAL RENAL FOSSA,YES,NORMAL,REDUCED
3,ABDUL ALIM,Data\ABDUL_ALIM_1_MONTHS.jpeg,P,2,NORMAL,NORMAL RENAL FOSSA,NO,NORMAL,REDUCED
4,ABDUL ALIM,Data\ABDUL_ALIM_1_MONTHS_1.jpeg,LPO,2,NORMAL,NORMAL RENAL FOSSA,NO,NORMAL,REDUCED


In [4]:
# CELL 4

groups = df[cfg.COL_PATIENT].astype(str).values

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=cfg.SEED)
train_idx, temp_idx = next(gss1.split(df, groups=groups))

df_train = df.iloc[train_idx].reset_index(drop=True)
df_temp = df.iloc[temp_idx].reset_index(drop=True)

groups_temp = df_temp[cfg.COL_PATIENT].astype(str).values
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=cfg.SEED)
val_idx, test_idx = next(gss2.split(df_temp, groups=groups_temp))

df_val = df_temp.iloc[val_idx].reset_index(drop=True)
df_test = df_temp.iloc[test_idx].reset_index(drop=True)

print("train:", df_train.shape, "val:", df_val.shape, "test:", df_test.shape)
print("TRAIN kidney count:\n", df_train[cfg.COL_KIDNEY_COUNT].value_counts().sort_index())
print("VAL kidney count:\n", df_val[cfg.COL_KIDNEY_COUNT].value_counts().sort_index())
print("TEST kidney count:\n", df_test[cfg.COL_KIDNEY_COUNT].value_counts().sort_index())


train: (1805, 23) val: (233, 23) test: (256, 23)
TRAIN kidney count:
 Kidney COUNT
1     347
2    1458
Name: count, dtype: int64
VAL kidney count:
 Kidney COUNT
1     33
2    200
Name: count, dtype: int64
TEST kidney count:
 Kidney COUNT
1     63
2    193
Name: count, dtype: int64


In [5]:
# CELL 5

def norm_str(x):
    return str(x).strip().upper()

# ANATOMY/POSITION vocabs (same rule as Phase-2)
anat_vals = sorted(df_train[cfg.COL_ANATOMY].map(norm_str).unique().tolist())
pos_vals  = sorted(df_train[cfg.COL_POSITION].map(norm_str).unique().tolist())

anat2idx = {v: i for i, v in enumerate(anat_vals)}
pos2idx  = {v: i for i, v in enumerate(pos_vals)}
idx2anat = {i: v for v, i in anat2idx.items()}
idx2pos  = {i: v for v, i in pos2idx.items()}

# Uptake vocab: we only train on NORMAL/REDUCED, and mask NOT APPLICABLE
UP_VALID = ["NORMAL", "REDUCED"]
up2idx = {v: i for i, v in enumerate(UP_VALID)}
idx2up = {i: v for v, i in up2idx.items()}

print("ANATOMY classes:", anat2idx)
print("POSITION classes:", pos2idx)
print("UPTAKE classes:", up2idx)

NUM_ANAT = len(anat2idx)
NUM_POS  = len(pos2idx)
NUM_UP   = len(up2idx)


ANATOMY classes: {'ECTOPIC': 0, 'HORSHOE': 1, 'NORMAL': 2, 'SINGLE': 3}
POSITION classes: {'ECTOPIC': 0, 'NORMAL RENAL FOSSA': 1}
UPTAKE classes: {'NORMAL': 0, 'REDUCED': 1}


In [6]:
# CELL 6

def load_image_from_link(link: str) -> Image.Image:
    p = Path(str(link))
    if not p.is_absolute():
        p = cfg.BASE_DIR / p
    if not p.exists():
        raise FileNotFoundError(f"Image path not found: {p}")
    return Image.open(p)

def body_box_crop(arr: np.ndarray) -> np.ndarray:
    thr = np.percentile(arr, cfg.BODY_Q)
    m = (arr > thr)
    if int(m.sum()) < int(cfg.BODY_MIN_PIXELS):
        return arr
    ys, xs = np.where(m)
    y0, y1 = int(ys.min()), int(ys.max())
    x0, x1 = int(xs.min()), int(xs.max())

    pad_y = int(round((y1 - y0 + 1) * 0.05))
    pad_x = int(round((x1 - x0 + 1) * 0.05))
    y0 = max(0, y0 - pad_y)
    y1 = min(arr.shape[0] - 1, y1 + pad_y)
    x0 = max(0, x0 - pad_x)
    x1 = min(arr.shape[1] - 1, x1 + pad_x)

    if (y1 - y0) < 10 or (x1 - x0) < 10:
        return arr
    return arr[y0:y1+1, x0:x1+1]

def layout_crop(arr: np.ndarray) -> np.ndarray:
    if not cfg.APPLY_LAYOUT_CROP:
        return arr
    h, w = arr.shape[:2]
    top = int(round(h * cfg.CROP_TOP_RATIO))
    bottom = h - int(round(h * cfg.CROP_BOTTOM_RATIO))
    left = int(round(w * cfg.CROP_LEFT_RATIO))
    right = w - int(round(w * cfg.CROP_RIGHT_RATIO))

    top = max(0, min(top, h - 2))
    bottom = max(top + 2, min(bottom, h))
    left = max(0, min(left, w - 2))
    right = max(left + 2, min(right, w))
    return arr[top:bottom, left:right]

def view_flip(arr: np.ndarray, view: str) -> np.ndarray:
    v = str(view).upper().strip()
    if v in ["RPO", "RLO"]:
        return np.fliplr(arr)
    return arr

def apply_preproc_uint8(arr: np.ndarray) -> np.ndarray:
    if cfg.INVERT_INTENSITY:
        arr = 255 - arr

    if cfg.APPLY_CLAHE and cv2 is not None:
        clahe = cv2.createCLAHE(clipLimit=cfg.CLAHE_CLIP, tileGridSize=(cfg.CLAHE_TILE, cfg.CLAHE_TILE))
        arr = clahe.apply(arr)

    if cfg.APPLY_DENOISE and cv2 is not None:
        arr = cv2.GaussianBlur(arr, (3, 3), 0)

    return arr

def make_roi_crop(arr_u8: np.ndarray) -> np.ndarray:
    if (not cfg.AUTO_ROI_CROP) or (cv2 is None):
        return arr_u8

    h, w = arr_u8.shape[:2]
    thr = np.percentile(arr_u8, cfg.HOT_Q)
    hot = (arr_u8 >= thr).astype(np.uint8) * 255
    hot = cv2.medianBlur(hot, 3)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(hot, connectivity=8)

    candidates = []
    for k in range(1, num_labels):
        x, y, ww, hh, area = stats[k]
        cx, cy = centroids[k]

        if area < cfg.HOT_MIN_AREA:
            continue
        if cy > cfg.BLADDER_CY_FRAC * h:
            continue
        if ww <= cfg.THIN_TALL_W and hh >= int(cfg.THIN_TALL_H_FRAC * h):
            continue

        candidates.append((area, x, y, ww, hh))

    if len(candidates) == 0:
        return arr_u8

    candidates.sort(key=lambda t: t[0], reverse=True)
    candidates = candidates[:cfg.HOT_MAX_KEEP]

    xs, ys, xe, ye = [], [], [], []
    for _, x, y, ww, hh in candidates:
        xs.append(x); ys.append(y)
        xe.append(x + ww - 1); ye.append(y + hh - 1)

    x0 = int(min(xs)); y0 = int(min(ys))
    x1 = int(max(xe)); y1 = int(max(ye))

    bh = max(1, y1 - y0 + 1)
    bw = max(1, x1 - x0 + 1)
    pad_y = int(round(bh * cfg.ROI_PAD_RATIO))
    pad_x = int(round(bw * cfg.ROI_PAD_RATIO))

    y0 = max(0, y0 - pad_y); y1 = min(h - 1, y1 + pad_y)
    x0 = max(0, x0 - pad_x); x1 = min(w - 1, x1 + pad_x)

    if (y1 - y0) < 10 or (x1 - x0) < 10:
        return arr_u8

    return arr_u8[y0:y1+1, x0:x1+1]

def preprocess_to_tensor(pil_img: Image.Image, view: str) -> torch.Tensor:
    arr = np.array(pil_img.convert("L"), dtype=np.uint8)
    arr = body_box_crop(arr)
    arr = layout_crop(arr)
    arr = view_flip(arr, view)
    arr = apply_preproc_uint8(arr)
    arr = make_roi_crop(arr)

    arr = np.array(Image.fromarray(arr).resize((cfg.IMG_SIZE, cfg.IMG_SIZE), resample=Image.BILINEAR), dtype=np.uint8)
    arr = arr.astype(np.float32) / 255.0
    return torch.from_numpy(arr).unsqueeze(0)


In [7]:
# CELL 7

def scar_to_label(s: str):
    v = norm_str(s)
    if v == "YES":
        return 1
    if v == "NO":
        return 0
    return -1

def uptake_to_label(s: str):
    v = norm_str(s)
    if v in up2idx:
        return up2idx[v]
    # NOT APPLICABLE -> invalid for loss
    return -1

class DMSAPhase3Dataset(Dataset):
    """
    Outputs:
      y_count: 0/1 (0=1kid, 1=2kid)
      y_anat:  class idx or -1
      y_pos:   class idx or -1
      y_scar:  0/1 or -1 (should always be valid in your CSV)
      y_up_l:  class idx or -1 (masked when NOT APPLICABLE)
      y_up_r:  class idx or -1 (masked when NOT APPLICABLE)

      masks: m_anat, m_pos, m_scar, m_up_l, m_up_r
    """
    def __init__(self, dfx: pd.DataFrame):
        self.dfx = dfx.reset_index(drop=True)

    def __len__(self):
        return len(self.dfx)

    def __getitem__(self, idx):
        row = self.dfx.iloc[idx]
        link = row[cfg.COL_LINK]
        view = row[cfg.COL_VIEW]

        kcount = int(row[cfg.COL_KIDNEY_COUNT])
        y_count = 0 if kcount == 1 else 1

        anat = norm_str(row[cfg.COL_ANATOMY])
        pos  = norm_str(row[cfg.COL_POSITION])

        y_anat = anat2idx.get(anat, -1)
        y_pos  = pos2idx.get(pos, -1)

        y_scar = scar_to_label(row[cfg.COL_SCAR])

        y_up_l = uptake_to_label(row[cfg.COL_UP_L])
        y_up_r = uptake_to_label(row[cfg.COL_UP_R])

        m_anat = 1 if y_anat >= 0 else 0
        m_pos  = 1 if y_pos >= 0 else 0
        m_scar = 1 if y_scar >= 0 else 0
        m_up_l = 1 if y_up_l >= 0 else 0
        m_up_r = 1 if y_up_r >= 0 else 0

        pil = load_image_from_link(link)
        x = preprocess_to_tensor(pil, view)

        return (
            x,
            torch.tensor(y_count, dtype=torch.long),
            torch.tensor(y_anat, dtype=torch.long),
            torch.tensor(y_pos, dtype=torch.long),
            torch.tensor(y_scar, dtype=torch.long),
            torch.tensor(y_up_l, dtype=torch.long),
            torch.tensor(y_up_r, dtype=torch.long),
            torch.tensor(m_anat, dtype=torch.float32),
            torch.tensor(m_pos, dtype=torch.float32),
            torch.tensor(m_scar, dtype=torch.float32),
            torch.tensor(m_up_l, dtype=torch.float32),
            torch.tensor(m_up_r, dtype=torch.float32),
        )


In [8]:
# CELL 8

device = torch.device(cfg.DEVICE)
print("Requested device:", cfg.DEVICE)
print("torch.cuda.is_available():", torch.cuda.is_available())
if device.type != "cuda":
    raise RuntimeError("CUDA NOT AVAILABLE — stop. Enable GPU first.")
print("CUDA device name:", torch.cuda.get_device_name(0))

train_ds = DMSAPhase3Dataset(df_train)
val_ds   = DMSAPhase3Dataset(df_val)
test_ds  = DMSAPhase3Dataset(df_test)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0)

print("Loaders:", len(train_loader), len(val_loader), len(test_loader))


Requested device: cuda
torch.cuda.is_available(): True
CUDA device name: NVIDIA GeForce GTX 1660 Ti
Loaders: 113 15 16


In [9]:
# CELL 9

class SharedEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        base = torchvision.models.resnet18(weights="DEFAULT")
        base.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.out_dim = 512

    def forward(self, x):
        z = self.backbone(x)
        return z.view(z.size(0), -1)

class CountHead(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.fc = nn.Linear(in_dim, 1)

    def forward(self, z):
        return self.fc(z).squeeze(1)

class ClassHead(nn.Module):
    def __init__(self, in_dim: int, num_classes: int):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)

    def forward(self, z):
        return self.fc(z)

class Phase3Model(nn.Module):
    def __init__(self, num_anat: int, num_pos: int, num_up: int):
        super().__init__()
        self.encoder = SharedEncoder()

        # existing Phase-2 heads
        self.count_head = CountHead(self.encoder.out_dim)
        self.anatomy_head = ClassHead(self.encoder.out_dim, num_anat)
        self.position_head = ClassHead(self.encoder.out_dim, num_pos)

        # new Phase-3 heads
        self.scar_head = CountHead(self.encoder.out_dim)     # BCE (YES/NO)
        self.up_l_head = ClassHead(self.encoder.out_dim, num_up)  # CE (NORMAL/REDUCED)
        self.up_r_head = ClassHead(self.encoder.out_dim, num_up)

    def forward(self, x):
        z = self.encoder(x)
        logit_count = self.count_head(z)
        logit_anat  = self.anatomy_head(z)
        logit_pos   = self.position_head(z)
        logit_scar  = self.scar_head(z)
        logit_up_l  = self.up_l_head(z)
        logit_up_r  = self.up_r_head(z)
        return logit_count, logit_anat, logit_pos, logit_scar, logit_up_l, logit_up_r

model = Phase3Model(NUM_ANAT, NUM_POS, NUM_UP).to(device)

# Load Phase-2 weights (encoder + count/anatomy/position)
ckpt_path = cfg.BASE_DIR / cfg.PHASE2_CKPT
sd = torch.load(str(ckpt_path), map_location=device)
model.load_state_dict(sd, strict=False)

print("Loaded Phase-2 ckpt:", ckpt_path)
print("Model on:", next(model.parameters()).device)

opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

# count imbalance like before
y_tr = df_train[cfg.COL_KIDNEY_COUNT].astype(int).values
n1 = int(np.sum(y_tr == 1))
n2 = int(np.sum(y_tr == 2))
pos_weight_count = torch.tensor([max(1e-6, n1 / max(1, n2))], dtype=torch.float32).to(device)
print("Train n1=", n1, "n2=", n2, "pos_weight_count(2kid)=", float(pos_weight_count.item()))

loss_count = nn.BCEWithLogitsLoss(pos_weight=pos_weight_count)
loss_ce = nn.CrossEntropyLoss()

# scar imbalance (YES/NO)
scar_tr = df_train[cfg.COL_SCAR].map(norm_str).values
n_no = int(np.sum(scar_tr == "NO"))
n_yes = int(np.sum(scar_tr == "YES"))
pos_weight_scar = torch.tensor([max(1e-6, n_no / max(1, n_yes))], dtype=torch.float32).to(device)
print("Train scar YES=", n_yes, "NO=", n_no, "pos_weight_scar(YES)=", float(pos_weight_scar.item()))
loss_scar = nn.BCEWithLogitsLoss(pos_weight=pos_weight_scar)


Loaded Phase-2 ckpt: F:\Bracu\THESIS\Final Defence\Dataset\code\phase2_count_anat_pos_best.pt
Model on: cuda:0
Train n1= 347 n2= 1458 pos_weight_count(2kid)= 0.23799726366996765
Train scar YES= 192 NO= 1613 pos_weight_scar(YES)= 8.401041984558105


In [10]:
# CELL 10

def train_one_epoch(model, loader):
    model.train()
    total = 0.0
    pbar = tqdm(loader, desc="TRAIN", leave=False)

    for batch in pbar:
        (
            x, y_count, y_anat, y_pos, y_scar, y_ul, y_ur,
            m_anat, m_pos, m_scar, m_ul, m_ur
        ) = batch

        x = x.to(device, non_blocking=True)

        y_count = y_count.float().to(device, non_blocking=True)
        y_anat  = y_anat.to(device, non_blocking=True)
        y_pos   = y_pos.to(device, non_blocking=True)

        y_scar  = y_scar.float().to(device, non_blocking=True)
        y_ul    = y_ul.to(device, non_blocking=True)
        y_ur    = y_ur.to(device, non_blocking=True)

        m_anat = m_anat.to(device, non_blocking=True)
        m_pos  = m_pos.to(device, non_blocking=True)
        m_scar = m_scar.to(device, non_blocking=True)
        m_ul   = m_ul.to(device, non_blocking=True)
        m_ur   = m_ur.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)

        logit_count, logit_anat, logit_pos, logit_scar, logit_ul, logit_ur = model(x)

        lc = loss_count(logit_count, y_count)

        la = torch.tensor(0.0, device=device)
        if m_anat.sum().item() > 0:
            idx = (m_anat > 0.5)
            la = loss_ce(logit_anat[idx], y_anat[idx])

        lp = torch.tensor(0.0, device=device)
        if m_pos.sum().item() > 0:
            idx = (m_pos > 0.5)
            lp = loss_ce(logit_pos[idx], y_pos[idx])

        ls = torch.tensor(0.0, device=device)
        if m_scar.sum().item() > 0:
            idx = (m_scar > 0.5)
            ls = loss_scar(logit_scar[idx], y_scar[idx])

        lul = torch.tensor(0.0, device=device)
        if m_ul.sum().item() > 0:
            idx = (m_ul > 0.5)
            lul = loss_ce(logit_ul[idx], y_ul[idx])

        lur = torch.tensor(0.0, device=device)
        if m_ur.sum().item() > 0:
            idx = (m_ur > 0.5)
            lur = loss_ce(logit_ur[idx], y_ur[idx])

        loss = (
            lc
            + cfg.W_ANATOMY * la
            + cfg.W_POSITION * lp
            + cfg.W_SCAR * ls
            + cfg.W_UP_L * lul
            + cfg.W_UP_R * lur
        )

        loss.backward()
        opt.step()

        total += float(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}", lc=f"{lc.item():.4f}")

    return total / max(1, len(loader))


@torch.no_grad()
def eval_loss(model, loader):
    model.eval()
    total = 0.0
    pbar = tqdm(loader, desc="VAL", leave=False)

    for batch in pbar:
        (
            x, y_count, y_anat, y_pos, y_scar, y_ul, y_ur,
            m_anat, m_pos, m_scar, m_ul, m_ur
        ) = batch

        x = x.to(device, non_blocking=True)

        y_count = y_count.float().to(device, non_blocking=True)
        y_anat  = y_anat.to(device, non_blocking=True)
        y_pos   = y_pos.to(device, non_blocking=True)

        y_scar  = y_scar.float().to(device, non_blocking=True)
        y_ul    = y_ul.to(device, non_blocking=True)
        y_ur    = y_ur.to(device, non_blocking=True)

        m_anat = m_anat.to(device, non_blocking=True)
        m_pos  = m_pos.to(device, non_blocking=True)
        m_scar = m_scar.to(device, non_blocking=True)
        m_ul   = m_ul.to(device, non_blocking=True)
        m_ur   = m_ur.to(device, non_blocking=True)

        logit_count, logit_anat, logit_pos, logit_scar, logit_ul, logit_ur = model(x)

        lc = loss_count(logit_count, y_count)

        la = torch.tensor(0.0, device=device)
        if m_anat.sum().item() > 0:
            idx = (m_anat > 0.5)
            la = loss_ce(logit_anat[idx], y_anat[idx])

        lp = torch.tensor(0.0, device=device)
        if m_pos.sum().item() > 0:
            idx = (m_pos > 0.5)
            lp = loss_ce(logit_pos[idx], y_pos[idx])

        ls = torch.tensor(0.0, device=device)
        if m_scar.sum().item() > 0:
            idx = (m_scar > 0.5)
            ls = loss_scar(logit_scar[idx], y_scar[idx])

        lul = torch.tensor(0.0, device=device)
        if m_ul.sum().item() > 0:
            idx = (m_ul > 0.5)
            lul = loss_ce(logit_ul[idx], y_ul[idx])

        lur = torch.tensor(0.0, device=device)
        if m_ur.sum().item() > 0:
            idx = (m_ur > 0.5)
            lur = loss_ce(logit_ur[idx], y_ur[idx])

        loss = (
            lc
            + cfg.W_ANATOMY * la
            + cfg.W_POSITION * lp
            + cfg.W_SCAR * ls
            + cfg.W_UP_L * lul
            + cfg.W_UP_R * lur
        )

        total += float(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}", lc=f"{lc.item():.4f}")

    return total / max(1, len(loader))


best = 1e18
pat = 0

for ep in range(1, cfg.EPOCHS + 1):
    tr = train_one_epoch(model, train_loader)
    va = eval_loss(model, val_loader)
    print(f"[PHASE3] epoch {ep}/{cfg.EPOCHS} train={tr:.4f} val={va:.4f}")

    if va < best - 1e-4:
        best = va
        pat = 0
        torch.save(model.state_dict(), str(cfg.BASE_DIR / "phase3_best.pt"))
    else:
        pat += 1
        if pat >= cfg.PATIENCE:
            print("[PHASE3] Early stopping")
            break

model.load_state_dict(torch.load(str(cfg.BASE_DIR / "phase3_best.pt"), map_location=device))
print("[PHASE3] Loaded best model | val loss:", best)


[PHASE3] epoch 1/12 train=0.7821 val=0.7491


[PHASE3] epoch 2/12 train=0.6018 val=0.7271


[PHASE3] epoch 3/12 train=0.4858 val=0.8336


[PHASE3] epoch 4/12 train=0.3880 val=0.8549


[PHASE3] epoch 5/12 train=0.2849 val=0.7657


[PHASE3] epoch 6/12 train=0.2210 val=0.8705
[PHASE3] Early stopping
[PHASE3] Loaded best model | val loss: 0.7271107912063599


In [11]:
# CELL 11 — Full evaluation across all trained heads (COUNT + ANATOMY + POSITION + SCAR + UPTAKE)

@torch.no_grad()
def predict_phase3(model, loader):
    """Compatibility helper used by earlier cells (returns uptake argmax labels)."""
    model.eval()

    p_count, y_count = [], []
    p_scar, y_scar, m_scar = [], [], []
    pred_ul, true_ul, m_ul = [], [], []
    pred_ur, true_ur, m_ur = [], [], []

    for batch in loader:
        (
            x, yc, ya, yp, ys, yul, yur,
            m_anat, m_pos, ms, mul, mur
        ) = batch

        x = x.to(device, non_blocking=True)
        logit_count, logit_anat, logit_pos, logit_scar, logit_ul, logit_ur = model(x)

        p_count.append(torch.sigmoid(logit_count).cpu().numpy())
        y_count.append(yc.numpy())

        p_scar.append(torch.sigmoid(logit_scar).cpu().numpy())
        y_scar.append(ys.numpy())
        m_scar.append(ms.numpy())

        pred_ul.append(torch.argmax(logit_ul, dim=1).cpu().numpy())
        true_ul.append(yul.numpy())
        m_ul.append(mul.numpy())

        pred_ur.append(torch.argmax(logit_ur, dim=1).cpu().numpy())
        true_ur.append(yur.numpy())
        m_ur.append(mur.numpy())

    p_count = np.concatenate(p_count, axis=0)
    y_count = np.concatenate(y_count, axis=0).astype(int)

    p_scar = np.concatenate(p_scar, axis=0)
    y_scar = np.concatenate(y_scar, axis=0).astype(int)
    m_scar = np.concatenate(m_scar, axis=0).astype(int)

    pred_ul = np.concatenate(pred_ul, axis=0)
    true_ul = np.concatenate(true_ul, axis=0).astype(int)
    m_ul = np.concatenate(m_ul, axis=0).astype(int)

    pred_ur = np.concatenate(pred_ur, axis=0)
    true_ur = np.concatenate(true_ur, axis=0).astype(int)
    m_ur = np.concatenate(m_ur, axis=0).astype(int)

    return p_count, y_count, p_scar, y_scar, m_scar, pred_ul, true_ul, m_ul, pred_ur, true_ur, m_ur


@torch.no_grad()
def predict_phase3_all(model, loader):
    """Collect predictions/labels for all heads, including masked ones."""
    model.eval()

    p_count, y_count = [], []

    pred_anat, true_anat, mask_anat = [], [], []
    pred_pos,  true_pos,  mask_pos  = [], [], []

    p_scar, y_scar, m_scar = [], [], []

    p_ul_red, y_ul, m_ul = [], [], []
    p_ur_red, y_ur, m_ur = [], [], []

    for batch in loader:
        (
            x, yc, ya, yp, ys, yul, yur,
            m_anat, m_pos, ms, mul, mur
        ) = batch

        x = x.to(device, non_blocking=True)
        logit_count, logit_anat, logit_pos, logit_scar, logit_ul, logit_ur = model(x)

        # COUNT (binary via sigmoid threshold)
        p_count.append(torch.sigmoid(logit_count).cpu().numpy())
        y_count.append(yc.numpy())

        # ANATOMY / POSITION (multiclass via argmax)
        pred_anat.append(torch.argmax(logit_anat, dim=1).cpu().numpy())
        true_anat.append(ya.numpy())
        mask_anat.append(m_anat.numpy())

        pred_pos.append(torch.argmax(logit_pos, dim=1).cpu().numpy())
        true_pos.append(yp.numpy())
        mask_pos.append(m_pos.numpy())

        # SCAR (binary via sigmoid threshold)
        p_scar.append(torch.sigmoid(logit_scar).cpu().numpy())
        y_scar.append(ys.numpy())
        m_scar.append(ms.numpy())

        # UPTAKE (probability of REDUCED = class index 1)
        p_ul_red.append(torch.softmax(logit_ul, dim=1)[:, 1].cpu().numpy())
        y_ul.append(yul.numpy())
        m_ul.append(mul.numpy())

        p_ur_red.append(torch.softmax(logit_ur, dim=1)[:, 1].cpu().numpy())
        y_ur.append(yur.numpy())
        m_ur.append(mur.numpy())

    # concat
    p_count = np.concatenate(p_count, axis=0)
    y_count = np.concatenate(y_count, axis=0).astype(int)

    pred_anat = np.concatenate(pred_anat, axis=0).astype(int)
    true_anat = np.concatenate(true_anat, axis=0).astype(int)
    mask_anat = np.concatenate(mask_anat, axis=0).astype(int)

    pred_pos = np.concatenate(pred_pos, axis=0).astype(int)
    true_pos = np.concatenate(true_pos, axis=0).astype(int)
    mask_pos = np.concatenate(mask_pos, axis=0).astype(int)

    p_scar = np.concatenate(p_scar, axis=0)
    y_scar = np.concatenate(y_scar, axis=0).astype(int)
    m_scar = np.concatenate(m_scar, axis=0).astype(int)

    p_ul_red = np.concatenate(p_ul_red, axis=0)
    y_ul = np.concatenate(y_ul, axis=0).astype(int)
    m_ul = np.concatenate(m_ul, axis=0).astype(int)

    p_ur_red = np.concatenate(p_ur_red, axis=0)
    y_ur = np.concatenate(y_ur, axis=0).astype(int)
    m_ur = np.concatenate(m_ur, axis=0).astype(int)

    return {
        "p_count": p_count, "y_count": y_count,
        "pred_anat": pred_anat, "true_anat": true_anat, "mask_anat": mask_anat,
        "pred_pos": pred_pos, "true_pos": true_pos, "mask_pos": mask_pos,
        "p_scar": p_scar, "y_scar": y_scar, "m_scar": m_scar,
        "p_ul_red": p_ul_red, "y_ul": y_ul, "m_ul": m_ul,
        "p_ur_red": p_ur_red, "y_ur": y_ur, "m_ur": m_ur,
    }


def _tune_binary_threshold(probs, y_true, mask=None, average="binary", pos_label=1):
    if mask is not None:
        mask = mask.astype(int)
        if mask.sum() == 0:
            return 0.5, -1.0
        y_true = y_true[mask == 1]
        probs = probs[mask == 1]

    best_thr, best_f1 = 0.5, -1.0
    for thr in np.linspace(0.1, 0.9, 17):
        pred = (probs >= thr).astype(int)
        f1 = f1_score(y_true, pred, average=average, pos_label=pos_label)
        if f1 > best_f1:
            best_f1, best_thr = f1, float(thr)
    return best_thr, best_f1


def _report_multiclass(name, y_true, y_pred, mask, idx2label):
    mask = mask.astype(int)
    if mask.sum() == 0:
        print(f"\n[{name}] No valid labels.")
        return

    yt = y_true[mask == 1]
    yp = y_pred[mask == 1]

    labels = list(range(len(idx2label)))
    target_names = [idx2label[i] for i in labels]

    print(f"\n[{name}] Valid samples: {int(mask.sum())}")
    print("macro_f1:", round(f1_score(yt, yp, average="macro"), 4),
          "balanced_acc:", round(balanced_accuracy_score(yt, yp), 4))
    print(confusion_matrix(yt, yp, labels=labels))
    print(classification_report(
        yt, yp,
        labels=labels,
        target_names=target_names,
        digits=4,
        zero_division=0,
    ))


@torch.no_grad()
def phase3_evaluate_all(model, val_loader, test_loader):
    # ----- VAL pass (for threshold tuning) -----
    val = predict_phase3_all(model, val_loader)

    # COUNT threshold on VAL (macro F1)
    count_thr, count_f1 = _tune_binary_threshold(val["p_count"], val["y_count"], average="macro")

    # SCAR threshold on VAL (F1 for YES class)
    scar_thr, scar_f1 = _tune_binary_threshold(
        val["p_scar"], val["y_scar"], mask=val["m_scar"], average="binary", pos_label=1
    )

    # UPTAKE thresholds on VAL (F1 for REDUCED class)
    ul_thr, ul_f1 = _tune_binary_threshold(
        val["p_ul_red"], val["y_ul"], mask=val["m_ul"], average="binary", pos_label=1
    )
    ur_thr, ur_f1 = _tune_binary_threshold(
        val["p_ur_red"], val["y_ur"], mask=val["m_ur"], average="binary", pos_label=1
    )

    print("=" * 72)
    print("PHASE-3: ALL TRAINED HEADS — VAL TUNING SUMMARY")
    print("=" * 72)
    print(f"COUNT threshold: {count_thr:.2f} | VAL macro_f1={count_f1:.4f}")
    if scar_f1 >= 0:
        print(f"SCAR threshold : {scar_thr:.2f} | VAL f1(YES)={scar_f1:.4f}")
    else:
        print("SCAR threshold : no valid SCAR labels on VAL")
    if ul_f1 >= 0:
        print(f"UP_L threshold : {ul_thr:.2f} | VAL f1(REDUCED)={ul_f1:.4f}")
    else:
        print("UP_L threshold : no valid uptake-left labels on VAL")
    if ur_f1 >= 0:
        print(f"UP_R threshold : {ur_thr:.2f} | VAL f1(REDUCED)={ur_f1:.4f}")
    else:
        print("UP_R threshold : no valid uptake-right labels on VAL")

    # ----- TEST pass (full reporting) -----
    te = predict_phase3_all(model, test_loader)

    # COUNT on TEST
    pred_count = (te["p_count"] >= count_thr).astype(int)
    cm = confusion_matrix(te["y_count"], pred_count, labels=[0, 1])
    rec0 = cm[0, 0] / max(1, (cm[0, 0] + cm[0, 1]))
    rec1 = cm[1, 1] / max(1, (cm[1, 0] + cm[1, 1]))

    print("\n" + "=" * 72)
    print("PHASE-3: ALL TRAINED HEADS — TEST RESULTS")
    print("=" * 72)

    print("\n[COUNT] TEST CM (0=1kid,1=2kid):\n", cm)
    print("COUNT recall(1kid)=", round(rec0, 4), "recall(2kid)=", round(rec1, 4))
    print("COUNT macro_f1:", round(f1_score(te["y_count"], pred_count, average="macro"), 4),
          "balanced_acc:", round(balanced_accuracy_score(te["y_count"], pred_count), 4))
    print(classification_report(te["y_count"], pred_count, digits=4, zero_division=0))

    # ANATOMY + POSITION on TEST (masked)
    _report_multiclass("ANATOMY", te["true_anat"], te["pred_anat"], te["mask_anat"], idx2anat)
    _report_multiclass("POSITION", te["true_pos"], te["pred_pos"], te["mask_pos"], idx2pos)

    # SCAR on TEST (masked)
    if te["m_scar"].sum() > 0:
        ys = te["y_scar"][te["m_scar"] == 1]
        ps = te["p_scar"][te["m_scar"] == 1]
        pred_s = (ps >= scar_thr).astype(int)
        print("\n[SCAR] Valid TEST samples:", int(te["m_scar"].sum()))
        print("[SCAR] TEST CM (0=NO,1=YES):\n", confusion_matrix(ys, pred_s, labels=[0, 1]))
        print(classification_report(ys, pred_s, digits=4, zero_division=0))
    else:
        print("\n[SCAR] No valid SCAR labels on TEST")

    # UPTAKE on TEST (masked)
    print("\n[UPTAKE LEFT] Valid TEST samples:", int(te["m_ul"].sum()), "| NOT APPLICABLE:", int(len(te["m_ul"]) - te["m_ul"].sum()))
    if te["m_ul"].sum() > 0:
        yl = te["y_ul"][te["m_ul"] == 1]
        pl = te["p_ul_red"][te["m_ul"] == 1]
        pred_l = (pl >= ul_thr).astype(int)
        labels_up = [0, 1]
        print(confusion_matrix(yl, pred_l, labels=labels_up))
        print(classification_report(
            yl, pred_l,
            labels=labels_up,
            digits=4,
            target_names=[idx2up[i] for i in range(NUM_UP)],
            zero_division=0,
        ))
    else:
        print("No valid uptake-left labels on TEST")

    print("\n[UPTAKE RIGHT] Valid TEST samples:", int(te["m_ur"].sum()), "| NOT APPLICABLE:", int(len(te["m_ur"]) - te["m_ur"].sum()))
    if te["m_ur"].sum() > 0:
        yr = te["y_ur"][te["m_ur"] == 1]
        pr = te["p_ur_red"][te["m_ur"] == 1]
        pred_r = (pr >= ur_thr).astype(int)
        labels_up = [0, 1]
        print(confusion_matrix(yr, pred_r, labels=labels_up))
        print(classification_report(
            yr, pred_r,
            labels=labels_up,
            digits=4,
            target_names=[idx2up[i] for i in range(NUM_UP)],
            zero_division=0,
        ))
    else:
        print("No valid uptake-right labels on TEST")

    print("=" * 72)

    return {
        "count_thr": float(count_thr),
        "scar_thr": float(scar_thr),
        "ul_thr": float(ul_thr),
        "ur_thr": float(ur_thr),
    }


# Run full evaluation (prints ANATOMY and POSITION too)
phase3_all_thresholds = phase3_evaluate_all(model, val_loader, test_loader)

# Keep compatibility with later cells that expect best_thr
best_thr = float(phase3_all_thresholds["count_thr"])
print("best_thr (COUNT):", best_thr)


PHASE-3: ALL TRAINED HEADS — VAL TUNING SUMMARY
COUNT threshold: 0.15 | VAL macro_f1=0.9564
SCAR threshold : 0.55 | VAL f1(YES)=0.4494
UP_L threshold : 0.55 | VAL f1(REDUCED)=0.7358
UP_R threshold : 0.35 | VAL f1(REDUCED)=0.5829

PHASE-3: ALL TRAINED HEADS — TEST RESULTS

[COUNT] TEST CM (0=1kid,1=2kid):
 [[ 60   3]
 [ 12 181]]
COUNT recall(1kid)= 0.9524 recall(2kid)= 0.9378
COUNT macro_f1: 0.9246 balanced_acc: 0.9451
              precision    recall  f1-score   support

           0     0.8333    0.9524    0.8889        63
           1     0.9837    0.9378    0.9602       193

    accuracy                         0.9414       256
   macro avg     0.9085    0.9451    0.9246       256
weighted avg     0.9467    0.9414    0.9427       256


[ANATOMY] Valid samples: 256
macro_f1: 0.6504 balanced_acc: 0.6459
[[ 18   0   0   8]
 [  0   0   2   1]
 [  0   0 154  10]
 [  1   0   2  60]]
              precision    recall  f1-score   support

     ECTOPIC     0.9474    0.6923    0.8000        

In [12]:
# CELL 12

run_info = {
    "phase": "phase3",
    "count_threshold": best_thr,
    "pos_weight_count_2kid": float(pos_weight_count.item()),
    "pos_weight_scar_yes": float(pos_weight_scar.item()),
    "train_n1": int(n1),
    "train_n2": int(n2),
    "num_anatomy_classes": int(NUM_ANAT),
    "num_position_classes": int(NUM_POS),
    "uptake_classes": UP_VALID,
    "w_anatomy": float(cfg.W_ANATOMY),
    "w_position": float(cfg.W_POSITION),
    "w_scar": float(cfg.W_SCAR),
    "w_up_l": float(cfg.W_UP_L),
    "w_up_r": float(cfg.W_UP_R),
    "img_size": int(cfg.IMG_SIZE),
    "phase2_ckpt_used": str(cfg.PHASE2_CKPT),
    "phase3_ckpt_saved": "phase3_best.pt",
}

out_path = cfg.BASE_DIR / "phase3_run_info.json"
pd.Series(run_info).to_json(out_path)
print("Saved:", out_path)

print("Next Phase-4: size via scale-based measurement + age table (hybrid rule-based).")


Saved: F:\Bracu\THESIS\Final Defence\Dataset\code\phase3_run_info.json
Next Phase-4: size via scale-based measurement + age table (hybrid rule-based).


In [13]:
# CELL 13 (ADD AS A NEW CELL)

@torch.no_grad()
def phase3_full_summary(model, val_loader, test_loader):
    # ---------- COUNT: tune threshold on VAL ----------
    p_val, yv, ps_v, ys_v, ms_v, pul_v, tul_v, mul_v, pur_v, tur_v, mur_v = predict_phase3(model, val_loader)

    best_thr_c = 0.5
    best_f1_c = -1
    for thr in np.linspace(0.1, 0.9, 17):
        pred = (p_val >= thr).astype(int)
        f1m = f1_score(yv, pred, average="macro")
        if f1m > best_f1_c:
            best_f1_c = f1m
            best_thr_c = float(thr)

    # ---------- SCAR: tune threshold on VAL ----------
    best_thr_s = 0.5
    best_f1_s = -1
    if ms_v.sum() > 0:
        ys2 = ys_v[ms_v == 1]
        ps2 = ps_v[ms_v == 1]
        for thr in np.linspace(0.1, 0.9, 17):
            pred = (ps2 >= thr).astype(int)
            f1p = f1_score(ys2, pred, average="binary", pos_label=1)
            if f1p > best_f1_s:
                best_f1_s = f1p
                best_thr_s = float(thr)

    print("=" * 70)
    print("PHASE-3 COMBINED SUMMARY")
    print("=" * 70)
    print(f"[COUNT] tuned threshold={best_thr_c:.2f} | VAL macro_f1={best_f1_c:.4f}")
    if best_f1_s >= 0:
        print(f"[SCAR ] tuned threshold={best_thr_s:.2f} | VAL f1(YES)={best_f1_s:.4f}")
    else:
        print("[SCAR ] No valid SCAR labels in VAL (unexpected).")

    # ---------- TEST ----------
    p_te, yt, ps, ys, ms, pul, tul, mul, pur, tur, mur = predict_phase3(model, test_loader)

    # COUNT test
    pred_c = (p_te >= best_thr_c).astype(int)
    cm_c = confusion_matrix(yt, pred_c)
    rec0 = cm_c[0,0] / max(1, (cm_c[0,0] + cm_c[0,1]))
    rec1 = cm_c[1,1] / max(1, (cm_c[1,0] + cm_c[1,1]))

    print("\n[COUNT] TEST CM (0=1kid,1=2kid):\n", cm_c)
    print("COUNT recall(1kid)=", round(rec0, 4), "recall(2kid)=", round(rec1, 4))
    print("COUNT macro_f1:", round(f1_score(yt, pred_c, average="macro"), 4),
          "balanced_acc:", round(balanced_accuracy_score(yt, pred_c), 4))
    print(classification_report(yt, pred_c, digits=4))

    # SCAR test
    if ms.sum() > 0:
        ys2 = ys[ms == 1]
        ps2 = ps[ms == 1]
        pred_s = (ps2 >= best_thr_s).astype(int)

        print("\n[SCAR] Valid TEST samples:", int(ms.sum()))
        print("[SCAR] TEST CM (0=NO,1=YES):\n", confusion_matrix(ys2, pred_s))
        print(classification_report(ys2, pred_s, digits=4))
    else:
        print("\n[SCAR] No valid SCAR labels in TEST (unexpected).")

    # Uptake Left
    print("\n[UPTAKE LEFT] Valid TEST samples:", int(mul.sum()), "| NOT APPLICABLE:", int(len(mul) - mul.sum()))
    if mul.sum() > 0:
        tul2 = tul[mul == 1]
        pul2 = pul[mul == 1]
        print(classification_report(tul2, pul2, digits=4, target_names=[idx2up[i] for i in range(NUM_UP)]))
    else:
        print("No valid uptake-left samples.")

    # Uptake Right
    print("\n[UPTAKE RIGHT] Valid TEST samples:", int(mur.sum()), "| NOT APPLICABLE:", int(len(mur) - mur.sum()))
    if mur.sum() > 0:
        tur2 = tur[mur == 1]
        pur2 = pur[mur == 1]
        print(classification_report(tur2, pur2, digits=4, target_names=[idx2up[i] for i in range(NUM_UP)]))
    else:
        print("No valid uptake-right samples.")

    print("=" * 70)

# Run it
phase3_full_summary(model, val_loader, test_loader)


PHASE-3 COMBINED SUMMARY
[COUNT] tuned threshold=0.15 | VAL macro_f1=0.9564
[SCAR ] tuned threshold=0.55 | VAL f1(YES)=0.4494

[COUNT] TEST CM (0=1kid,1=2kid):
 [[ 60   3]
 [ 12 181]]
COUNT recall(1kid)= 0.9524 recall(2kid)= 0.9378
COUNT macro_f1: 0.9246 balanced_acc: 0.9451
              precision    recall  f1-score   support

           0     0.8333    0.9524    0.8889        63
           1     0.9837    0.9378    0.9602       193

    accuracy                         0.9414       256
   macro avg     0.9085    0.9451    0.9246       256
weighted avg     0.9467    0.9414    0.9427       256


[SCAR] Valid TEST samples: 256
[SCAR] TEST CM (0=NO,1=YES):
 [[198  43]
 [  8   7]]
              precision    recall  f1-score   support

           0     0.9612    0.8216    0.8859       241
           1     0.1400    0.4667    0.2154        15

    accuracy                         0.8008       256
   macro avg     0.5506    0.6441    0.5506       256
weighted avg     0.9130    0.8008    0.8

In [14]:
# CELL 14 — Threshold tuning for SCAR and UPTAKE (VAL only) [FIXED + probability-based]

@torch.no_grad()
def _predict_phase3_probabilities(model, loader):
    """Return probability-style outputs needed for threshold tuning."""
    model.eval()

    p_scar, y_scar, m_scar = [], [], []
    p_ul_red, y_ul, m_ul = [], [], []
    p_ur_red, y_ur, m_ur = [], [], []

    for batch in loader:
        (
            x, yc, ya, yp, ys, yul, yur,
            m_anat, m_pos, ms, mul, mur
        ) = batch

        x = x.to(device, non_blocking=True)
        logit_count, logit_anat, logit_pos, logit_scar, logit_ul, logit_ur = model(x)

        p_scar.append(torch.sigmoid(logit_scar).cpu().numpy())
        y_scar.append(ys.numpy())
        m_scar.append(ms.numpy())

        p_ul_red.append(torch.softmax(logit_ul, dim=1)[:, 1].cpu().numpy())
        y_ul.append(yul.numpy())
        m_ul.append(mul.numpy())

        p_ur_red.append(torch.softmax(logit_ur, dim=1)[:, 1].cpu().numpy())
        y_ur.append(yur.numpy())
        m_ur.append(mur.numpy())

    p_scar = np.concatenate(p_scar, axis=0)
    y_scar = np.concatenate(y_scar, axis=0).astype(int)
    m_scar = np.concatenate(m_scar, axis=0).astype(int)

    p_ul_red = np.concatenate(p_ul_red, axis=0)
    y_ul = np.concatenate(y_ul, axis=0).astype(int)
    m_ul = np.concatenate(m_ul, axis=0).astype(int)

    p_ur_red = np.concatenate(p_ur_red, axis=0)
    y_ur = np.concatenate(y_ur, axis=0).astype(int)
    m_ur = np.concatenate(m_ur, axis=0).astype(int)

    return p_scar, y_scar, m_scar, p_ul_red, y_ul, m_ul, p_ur_red, y_ur, m_ur


@torch.no_grad()
def tune_phase3_thresholds(model, val_loader):
    (
        p_scar, y_scar, m_scar,
        p_ul_red, y_ul, m_ul,
        p_ur_red, y_ur, m_ur,
    ) = _predict_phase3_probabilities(model, val_loader)

    # ---- SCAR threshold (YES = 1) ----
    best_scar_thr, best_scar_f1 = 0.5, -1.0
    if m_scar.sum() > 0:
        ys = y_scar[m_scar == 1]
        ps = p_scar[m_scar == 1]
        for thr in np.linspace(0.1, 0.9, 17):
            pred = (ps >= thr).astype(int)
            f1 = f1_score(ys, pred, average="binary", pos_label=1)
            if f1 > best_scar_f1:
                best_scar_f1, best_scar_thr = f1, float(thr)

    # ---- Uptake LEFT threshold (REDUCED = 1) ----
    best_ul_thr, best_ul_f1 = 0.5, -1.0
    if m_ul.sum() > 0:
        yl = y_ul[m_ul == 1]
        pl = p_ul_red[m_ul == 1]
        for thr in np.linspace(0.1, 0.9, 17):
            pred = (pl >= thr).astype(int)
            f1 = f1_score(yl, pred, average="binary", pos_label=1)
            if f1 > best_ul_f1:
                best_ul_f1, best_ul_thr = f1, float(thr)

    # ---- Uptake RIGHT threshold (REDUCED = 1) ----
    best_ur_thr, best_ur_f1 = 0.5, -1.0
    if m_ur.sum() > 0:
        yr = y_ur[m_ur == 1]
        pr = p_ur_red[m_ur == 1]
        for thr in np.linspace(0.1, 0.9, 17):
            pred = (pr >= thr).astype(int)
            f1 = f1_score(yr, pred, average="binary", pos_label=1)
            if f1 > best_ur_f1:
                best_ur_f1, best_ur_thr = f1, float(thr)

    print("=== Phase-3 Tuned Thresholds (VAL) ===")
    print(f"SCAR YES threshold        : {best_scar_thr:.2f} (f1={best_scar_f1:.3f})")
    print(f"UPTAKE LEFT RED threshold : {best_ul_thr:.2f} (f1={best_ul_f1:.3f})")
    print(f"UPTAKE RIGHT RED threshold: {best_ur_thr:.2f} (f1={best_ur_f1:.3f})")

    return {
        "scar_thr": float(best_scar_thr),
        "ul_thr": float(best_ul_thr),
        "ur_thr": float(best_ur_thr),
    }


phase3_thresholds = tune_phase3_thresholds(model, val_loader)


=== Phase-3 Tuned Thresholds (VAL) ===
SCAR YES threshold        : 0.55 (f1=0.449)
UPTAKE LEFT RED threshold : 0.55 (f1=0.736)
UPTAKE RIGHT RED threshold: 0.35 (f1=0.583)


In [15]:
# CELL — Medical Report Generator (FINAL, laterality-aware)

def generate_medical_report(
    preds,
    thresholds,
    count_thr=0.20
):
    report = {}

    # ---------- KIDNEY COUNT ----------
    kidney_count = 2 if preds["count_prob"] >= count_thr else 1
    report["kidney_count"] = kidney_count

    # ---------- POSITION & ANATOMY ----------
    report["position"] = preds["position"]
    report["anatomy"] = preds["anatomy"]

    # ---------- SCAR ----------
    scar_yes = preds["scar_prob"] >= thresholds["scar_thr"]
    report["scar"] = "YES" if scar_yes else "NO"
    report["scar_conf"] = round(float(preds["scar_prob"]), 3)

    # ---------- CORTICAL UPTAKE ----------
    ul_prob = preds["ul_prob"]
    ur_prob = preds["ur_prob"]

    if kidney_count == 2:
        # both kidneys exist
        report["uptake_left"]  = "REDUCED" if ul_prob >= thresholds["ul_thr"] else "NORMAL"
        report["uptake_right"] = "REDUCED" if ur_prob >= thresholds["ur_thr"] else "NORMAL"

    else:
        # exactly ONE kidney exists → infer laterality
        if ul_prob >= ur_prob:
            present_side = "LEFT"
        else:
            present_side = "RIGHT"

        if present_side == "LEFT":
            report["uptake_left"]  = "REDUCED" if ul_prob >= thresholds["ul_thr"] else "NORMAL"
            report["uptake_right"] = "NOT APPLICABLE"
        else:
            report["uptake_left"]  = "NOT APPLICABLE"
            report["uptake_right"] = "REDUCED" if ur_prob >= thresholds["ur_thr"] else "NORMAL"

        report["single_kidney_side"] = present_side

    # ---------- CONSISTENCY FLAGS ----------
    flags = []

    if kidney_count == 1:
        if report["uptake_left"] == "NOT APPLICABLE" and report["uptake_right"] == "NOT APPLICABLE":
            flags.append("Invalid state: single kidney but no uptake reported")

    if report["scar"] == "YES":
        if (
            report.get("uptake_left") == "NORMAL" and
            report.get("uptake_right") == "NORMAL"
        ):
            flags.append("Scar predicted but uptake appears normal")

    report["flags"] = flags

    return report


In [16]:
# CELL 16 — Example inference + generated medical report (CORRECT for tuple outputs)

model.eval()
idx = 0  # change index to inspect different cases

# get one batch safely (dataset returns many items)
batch = next(iter(test_loader))

# FIRST element is always image tensor in your pipeline
x_batch = batch[0]
x = x_batch[idx:idx+1].to(device)

with torch.no_grad():
    # your Phase-3 model returns a tuple:
    # (logit_count, logit_anat, logit_pos, logit_scar, logit_up_l, logit_up_r)
    logit_count, logit_anat, logit_pos, logit_scar, logit_ul, logit_ur = model(x)

# convert to probabilities / predicted labels
count_prob = torch.sigmoid(logit_count).item()   # prob of class "2 kidneys" (label 1)
scar_prob  = torch.sigmoid(logit_scar).item()    # prob of scar=YES

# anatomy / position are multiclass logits
pos_logits  = logit_pos.detach().cpu().numpy()[0]
anat_logits = logit_anat.detach().cpu().numpy()[0]

# uptake logits -> softmax prob for REDUCED (index 1)
ul_prob = torch.softmax(logit_ul, dim=1)[0, 1].item()
ur_prob = torch.softmax(logit_ur, dim=1)[0, 1].item()

sample_preds = {
    "count_prob": float(count_prob),
    "position": idx2pos[int(np.argmax(pos_logits))],
    "anatomy": idx2anat[int(np.argmax(anat_logits))],
    "scar_prob": float(scar_prob),
    "ul_prob": float(ul_prob),
    "ur_prob": float(ur_prob),
}

report = generate_medical_report(sample_preds, phase3_thresholds, count_thr=best_thr)

print("===== GENERATED MEDICAL REPORT =====")
for k, v in report.items():
    print(f"{k}: {v}")


===== GENERATED MEDICAL REPORT =====
kidney_count: 1
position: NORMAL RENAL FOSSA
anatomy: SINGLE
scar: NO
scar_conf: 0.27
uptake_left: NORMAL
uptake_right: NOT APPLICABLE
single_kidney_side: LEFT
flags: []


In [17]:
# CELL ? Full evaluation for all trained heads (COUNT + ANATOMY + POSITION + SCAR + UPTAKE)

@torch.no_grad()
def predict_phase3_all(model, loader):
    model.eval()

    p_count, y_count = [], []

    pred_anat, true_anat, mask_anat = [], [], []
    pred_pos,  true_pos,  mask_pos  = [], [], []

    p_scar, y_scar, m_scar = [], [], []

    p_ul_red, y_ul, m_ul = [], [], []
    p_ur_red, y_ur, m_ur = [], [], []

    for batch in loader:
        (
            x, yc, ya, yp, ys, yul, yur,
            m_anat, m_pos, ms, mul, mur
        ) = batch

        x = x.to(device, non_blocking=True)
        logit_count, logit_anat, logit_pos, logit_scar, logit_ul, logit_ur = model(x)

        # COUNT (binary via sigmoid threshold)
        p_count.append(torch.sigmoid(logit_count).cpu().numpy())
        y_count.append(yc.numpy())

        # ANATOMY / POSITION (multiclass via argmax)
        pred_anat.append(torch.argmax(logit_anat, dim=1).cpu().numpy())
        true_anat.append(ya.numpy())
        mask_anat.append(m_anat.numpy())

        pred_pos.append(torch.argmax(logit_pos, dim=1).cpu().numpy())
        true_pos.append(yp.numpy())
        mask_pos.append(m_pos.numpy())

        # SCAR (binary via sigmoid threshold)
        p_scar.append(torch.sigmoid(logit_scar).cpu().numpy())
        y_scar.append(ys.numpy())
        m_scar.append(ms.numpy())

        # UPTAKE (track REDUCED probability + true label + mask)
        p_ul_red.append(torch.softmax(logit_ul, dim=1)[:, 1].cpu().numpy())
        y_ul.append(yul.numpy())
        m_ul.append(mul.numpy())

        p_ur_red.append(torch.softmax(logit_ur, dim=1)[:, 1].cpu().numpy())
        y_ur.append(yur.numpy())
        m_ur.append(mur.numpy())

    # concat
    p_count = np.concatenate(p_count, axis=0)
    y_count = np.concatenate(y_count, axis=0).astype(int)

    pred_anat = np.concatenate(pred_anat, axis=0).astype(int)
    true_anat = np.concatenate(true_anat, axis=0).astype(int)
    mask_anat = np.concatenate(mask_anat, axis=0).astype(int)

    pred_pos = np.concatenate(pred_pos, axis=0).astype(int)
    true_pos = np.concatenate(true_pos, axis=0).astype(int)
    mask_pos = np.concatenate(mask_pos, axis=0).astype(int)

    p_scar = np.concatenate(p_scar, axis=0)
    y_scar = np.concatenate(y_scar, axis=0).astype(int)
    m_scar = np.concatenate(m_scar, axis=0).astype(int)

    p_ul_red = np.concatenate(p_ul_red, axis=0)
    y_ul = np.concatenate(y_ul, axis=0).astype(int)
    m_ul = np.concatenate(m_ul, axis=0).astype(int)

    p_ur_red = np.concatenate(p_ur_red, axis=0)
    y_ur = np.concatenate(y_ur, axis=0).astype(int)
    m_ur = np.concatenate(m_ur, axis=0).astype(int)

    return {
        "p_count": p_count, "y_count": y_count,
        "pred_anat": pred_anat, "true_anat": true_anat, "mask_anat": mask_anat,
        "pred_pos": pred_pos, "true_pos": true_pos, "mask_pos": mask_pos,
        "p_scar": p_scar, "y_scar": y_scar, "m_scar": m_scar,
        "p_ul_red": p_ul_red, "y_ul": y_ul, "m_ul": m_ul,
        "p_ur_red": p_ur_red, "y_ur": y_ur, "m_ur": m_ur,
    }


def _tune_binary_threshold(probs, y_true, mask=None, average="binary", pos_label=1):
    if mask is not None:
        mask = mask.astype(int)
        if mask.sum() == 0:
            return 0.5, -1.0
        y_true = y_true[mask == 1]
        probs = probs[mask == 1]

    best_thr, best_f1 = 0.5, -1.0
    for thr in np.linspace(0.1, 0.9, 17):
        pred = (probs >= thr).astype(int)
        f1 = f1_score(y_true, pred, average=average, pos_label=pos_label)
        if f1 > best_f1:
            best_f1, best_thr = f1, float(thr)
    return best_thr, best_f1


def _report_multiclass(name, y_true, y_pred, mask, idx2label):
    mask = mask.astype(int)
    if mask.sum() == 0:
        print(f"\n[{name}] No valid labels.")
        return

    yt = y_true[mask == 1]
    yp = y_pred[mask == 1]

    labels = list(range(len(idx2label)))
    target_names = [idx2label[i] for i in labels]

    print(f"\n[{name}] Valid samples: {int(mask.sum())}")
    print("macro_f1:", round(f1_score(yt, yp, average="macro"), 4),
          "balanced_acc:", round(balanced_accuracy_score(yt, yp), 4))
    print(confusion_matrix(yt, yp, labels=labels))
    print(classification_report(
        yt, yp,
        labels=labels,
        target_names=target_names,
        digits=4,
        zero_division=0,
    ))


@torch.no_grad()
def phase3_evaluate_all(model, val_loader, test_loader):
    # ----- VAL pass (for threshold tuning) -----
    val = predict_phase3_all(model, val_loader)

    # COUNT threshold on VAL (macro F1)
    count_thr, count_f1 = _tune_binary_threshold(val["p_count"], val["y_count"], average="macro")

    # SCAR threshold on VAL (F1 for YES class)
    scar_thr, scar_f1 = _tune_binary_threshold(val["p_scar"], val["y_scar"], mask=val["m_scar"], average="binary", pos_label=1)

    # UPTAKE thresholds on VAL (F1 for REDUCED class)
    ul_thr, ul_f1 = _tune_binary_threshold(val["p_ul_red"], val["y_ul"], mask=val["m_ul"], average="binary", pos_label=1)
    ur_thr, ur_f1 = _tune_binary_threshold(val["p_ur_red"], val["y_ur"], mask=val["m_ur"], average="binary", pos_label=1)

    print("=" * 72)
    print("PHASE-3: ALL TRAINED HEADS ? VAL TUNING SUMMARY")
    print("=" * 72)
    print(f"COUNT threshold: {count_thr:.2f} | VAL macro_f1={count_f1:.4f}")
    if scar_f1 >= 0:
        print(f"SCAR threshold : {scar_thr:.2f} | VAL f1(YES)={scar_f1:.4f}")
    else:
        print("SCAR threshold : no valid SCAR labels on VAL")
    if ul_f1 >= 0:
        print(f"UP_L threshold : {ul_thr:.2f} | VAL f1(REDUCED)={ul_f1:.4f}")
    else:
        print("UP_L threshold : no valid uptake-left labels on VAL")
    if ur_f1 >= 0:
        print(f"UP_R threshold : {ur_thr:.2f} | VAL f1(REDUCED)={ur_f1:.4f}")
    else:
        print("UP_R threshold : no valid uptake-right labels on VAL")

    # ----- TEST pass (full reporting) -----
    te = predict_phase3_all(model, test_loader)

    # COUNT on TEST
    pred_count = (te["p_count"] >= count_thr).astype(int)
    cm = confusion_matrix(te["y_count"], pred_count, labels=[0, 1])
    rec0 = cm[0, 0] / max(1, (cm[0, 0] + cm[0, 1]))
    rec1 = cm[1, 1] / max(1, (cm[1, 0] + cm[1, 1]))

    print("\n" + "=" * 72)
    print("PHASE-3: ALL TRAINED HEADS ? TEST RESULTS")
    print("=" * 72)

    print("\n[COUNT] TEST CM (0=1kid,1=2kid):\n", cm)
    print("COUNT recall(1kid)=", round(rec0, 4), "recall(2kid)=", round(rec1, 4))
    print("COUNT macro_f1:", round(f1_score(te["y_count"], pred_count, average="macro"), 4),
          "balanced_acc:", round(balanced_accuracy_score(te["y_count"], pred_count), 4))
    print(classification_report(te["y_count"], pred_count, digits=4, zero_division=0))

    # ANATOMY + POSITION on TEST (masked)
    _report_multiclass("ANATOMY", te["true_anat"], te["pred_anat"], te["mask_anat"], idx2anat)
    _report_multiclass("POSITION", te["true_pos"], te["pred_pos"], te["mask_pos"], idx2pos)

    # SCAR on TEST (masked)
    if te["m_scar"].sum() > 0:
        ys = te["y_scar"][te["m_scar"] == 1]
        ps = te["p_scar"][te["m_scar"] == 1]
        pred_s = (ps >= scar_thr).astype(int)
        print("\n[SCAR] Valid TEST samples:", int(te["m_scar"].sum()))
        print("[SCAR] TEST CM (0=NO,1=YES):\n", confusion_matrix(ys, pred_s, labels=[0, 1]))
        print(classification_report(ys, pred_s, digits=4, zero_division=0))
    else:
        print("\n[SCAR] No valid SCAR labels on TEST")

    # UPTAKE on TEST (masked)
    print("\n[UPTAKE LEFT] Valid TEST samples:", int(te["m_ul"].sum()), "| NOT APPLICABLE:", int(len(te["m_ul"]) - te["m_ul"].sum()))
    if te["m_ul"].sum() > 0:
        yl = te["y_ul"][te["m_ul"] == 1]
        pl = te["p_ul_red"][te["m_ul"] == 1]
        pred_l = (pl >= ul_thr).astype(int)
        labels_up = [0, 1]
        print(confusion_matrix(yl, pred_l, labels=labels_up))
        print(classification_report(yl, pred_l, labels=labels_up, digits=4, target_names=[idx2up[i] for i in range(NUM_UP)], zero_division=0))
    else:
        print("No valid uptake-left labels on TEST")

    print("\n[UPTAKE RIGHT] Valid TEST samples:", int(te["m_ur"].sum()), "| NOT APPLICABLE:", int(len(te["m_ur"]) - te["m_ur"].sum()))
    if te["m_ur"].sum() > 0:
        yr = te["y_ur"][te["m_ur"] == 1]
        pr = te["p_ur_red"][te["m_ur"] == 1]
        pred_r = (pr >= ur_thr).astype(int)
        labels_up = [0, 1]
        print(confusion_matrix(yr, pred_r, labels=labels_up))
        print(classification_report(yr, pred_r, labels=labels_up, digits=4, target_names=[idx2up[i] for i in range(NUM_UP)], zero_division=0))
    else:
        print("No valid uptake-right labels on TEST")

    print("=" * 72)

    return {
        "count_thr": count_thr,
        "scar_thr": scar_thr,
        "ul_thr": ul_thr,
        "ur_thr": ur_thr,
    }


# Run full evaluation (prints ANATOMY and POSITION too)
phase3_all_thresholds = phase3_evaluate_all(model, val_loader, test_loader)


PHASE-3: ALL TRAINED HEADS ? VAL TUNING SUMMARY
COUNT threshold: 0.15 | VAL macro_f1=0.9564
SCAR threshold : 0.55 | VAL f1(YES)=0.4494
UP_L threshold : 0.55 | VAL f1(REDUCED)=0.7358
UP_R threshold : 0.35 | VAL f1(REDUCED)=0.5829

PHASE-3: ALL TRAINED HEADS ? TEST RESULTS

[COUNT] TEST CM (0=1kid,1=2kid):
 [[ 60   3]
 [ 12 181]]
COUNT recall(1kid)= 0.9524 recall(2kid)= 0.9378
COUNT macro_f1: 0.9246 balanced_acc: 0.9451
              precision    recall  f1-score   support

           0     0.8333    0.9524    0.8889        63
           1     0.9837    0.9378    0.9602       193

    accuracy                         0.9414       256
   macro avg     0.9085    0.9451    0.9246       256
weighted avg     0.9467    0.9414    0.9427       256


[ANATOMY] Valid samples: 256
macro_f1: 0.6504 balanced_acc: 0.6459
[[ 18   0   0   8]
 [  0   0   2   1]
 [  0   0 154  10]
 [  1   0   2  60]]
              precision    recall  f1-score   support

     ECTOPIC     0.9474    0.6923    0.8000        